In [435]:
import os
import sys
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime

## Читаем файл

In [434]:
deals_path = r"C:\Users\Admin\Desktop\ICH\Учеба, курсы в записи, домашки\Финальный проект\Deals (Done).xlsx"
deals = pd.read_excel(deals_path, dtype={"Id": "string", "Contact Name": "string"})

In [436]:
# Приводим названия колонок к удобному формату
deals.columns = (deals.columns.str.strip().str.lower().str.replace(" ", "_", regex=False))
deals = deals.rename(columns={"contact_name": "contact_id"})
deals.shape

(21595, 23)

In [437]:
deals.head()

,id,deal_owner_name,closing_date,quality,stage,lost_reason,page,campaign,sla,content,...,product,education_type,created_time,course_duration,months_of_study,initial_amount_paid,offer_total_amount,contact_id,city,level_of_deutsch
0,5805028000056864695,Ben Hall,NaN,NaN,New Lead,NaN,/eng/test,03.07.23women,NaN,v16,...,NaN,NaN,21.06.2024 15:30,NaN,NaN,NaN,NaN,5805028000056849495,NaN,NaN
1,5805028000056859489,Ulysses Adams,NaN,NaN,New Lead,NaN,/at-eng,NaN,NaN,NaN,...,Web Developer,Morning,21.06.2024 15:23,6.0,NaN,0,2000,5805028000056834471,NaN,NaN
2,5805028000056832357,Ulysses Adams,21.06.2024,D - Non Target,Lost,Non target,/at-eng,engwien_AT,00:26:43,b1-at,...,NaN,NaN,21.06.2024 14:45,NaN,NaN,NaN,NaN,5805028000056854421,NaN,NaN
3,5805028000056824246,Eva Kent,21.06.2024,E - Non Qualified,Lost,Invalid number,/eng,04.07.23recentlymoved_DE,01:00:04,bloggersvideo14com,...,NaN,NaN,21.06.2024 13:32,NaN,NaN,NaN,NaN,5805028000056889351,NaN,NaN
4,5805028000056873292,Ben Hall,21.06.2024,D - Non Target,Lost,Non target,/eng,discovery_DE,00:53:12,website,...,NaN,NaN,21.06.2024 13:21,NaN,NaN,NaN,NaN,5805028000056876176,NaN,NaN


In [438]:
deals.dtypes

id                     string[python]
deal_owner_name                object
closing_date                   object
quality                        object
stage                          object
lost_reason                    object
page                           object
campaign                       object
sla                            object
content                        object
term                           object
source                         object
payment_type                   object
product                        object
education_type                 object
created_time                   object
course_duration               float64
months_of_study               float64
initial_amount_paid            object
offer_total_amount             object
contact_id             string[python]
city                           object
level_of_deutsch               object
dtype: object

In [439]:
deals.isna().sum()

id                         2
deal_owner_name           31
closing_date            6950
quality                 2255
stage                      2
lost_reason             5471
page                       2
campaign                5528
sla                     6062
content                 7448
term                    9141
source                     2
payment_type           21099
product                18003
education_type         18295
created_time               2
course_duration        18008
months_of_study        20755
initial_amount_paid    17430
offer_total_amount     17410
contact_id                63
city                   19084
level_of_deutsch       20344
dtype: int64

In [440]:
# Общая статистика по заполненности, типам и уникальности
deals_info = pd.DataFrame({
    "column": deals.columns,
    "non_null": deals.notna().sum().values,
    "missing": deals.isna().sum().values,
    "missing_pct": (deals.isna().mean().values * 100).round(2),
    "dtype": deals.dtypes.astype(str).values,
    "unique_values": deals.nunique(dropna=True).values})
deals_info

,column,non_null,missing,missing_pct,dtype,unique_values
0,id,21593,2,0.01,string,21593
1,deal_owner_name,21564,31,0.14,object,27
2,closing_date,14645,6950,32.18,object,359
3,quality,19340,2255,10.44,object,6
4,stage,21593,2,0.01,object,13
5,lost_reason,16124,5471,25.33,object,21
6,page,21593,2,0.01,object,34
7,campaign,16067,5528,25.60,object,154
8,sla,15533,6062,28.07,object,13357
9,content,14147,7448,34.49,object,187


### Проверяем ключевые ID

In [441]:
print("Строк всего:", len(deals))
print("Пропусков в id:", deals["id"].isna().sum())
print("Уникальных id:", deals["id"].nunique(dropna=True))
print("Дубликатов по id:", deals["id"].duplicated().sum())

print("Пропусков в contact_id:", deals["contact_id"].isna().sum())
print("Уникальных contact_id:", deals["contact_id"].nunique(dropna=True))

Строк всего: 21595
Пропусков в id: 2
Уникальных id: 21593
Дубликатов по id: 1
Пропусков в contact_id: 63
Уникальных contact_id: 18089


In [442]:
# Смотрим ключевые категориальные поля
key_categorical_columns = [
    "stage",
    "lost_reason",
    "source",
    "quality",
    "payment_type",
    "product",
    "education_type",
    "city",
    "level_of_deutsch"]

for col in key_categorical_columns:
    print(f"\n{col}")
    print("-" * 50)
    print(deals[col].value_counts(dropna=False).head(20))


stage
--------------------------------------------------
stage
Lost                         15743
Call Delayed                  2248
Registered on Webinar         2072
Payment Done                   858
Waiting For Payment            325
Qualificated                   128
Registered on Offline Day      100
Need to Call - Sales            33
Need To Call                    31
Test Sent                       25
Need a consultation             23
New Lead                         6
NaN                              2
Free Education                   1
Name: count, dtype: int64

lost_reason
--------------------------------------------------
lost_reason
NaN                                        5471
Doesn't Answer                             4135
Changed Decision                           2146
Duplicate                                  1771
Non target                                 1761
Stopped Answering                          1588
Invalid number                             1481
needs ti

### Удаляем технически пустые строки

In [443]:
# В есть строки без id — это технический мусор, так как в них вообще нет данных.
deals_without_id = deals[deals["id"].isna()]
deals_without_id

,id,deal_owner_name,closing_date,quality,stage,lost_reason,page,campaign,sla,content,...,product,education_type,created_time,course_duration,months_of_study,initial_amount_paid,offer_total_amount,contact_id,city,level_of_deutsch
21593,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN
21594,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,#REF!,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN


In [444]:
n_before_empty_id = len(deals)
deals = deals.dropna(subset=["id"]).reset_index(drop=True)

print("Строк до удаления пустых id:", n_before_empty_id)
print("Строк после удаления пустых id:", len(deals))
print("Удалено строк:", n_before_empty_id - len(deals))

Строк до удаления пустых id: 21595
Строк после удаления пустых id: 21593
Удалено строк: 2


### Применяем mapping контактов из очищенной таблицы Contacts
В deals остались contact_id, которых уже нет в contacts.
Причина: это старые contact_id из contacts_duplicates_mapping.csv. То есть в Contacts удалили дубли и создали mapping, в Calls mapping применили, а в Deals часть старых contact_id осталась.

In [445]:
contacts_mapping_path = r"C:\Users\Admin\Desktop\ICH\Учеба, курсы в записи, домашки\Финальный проект\дополнительные файлы\contacts_duplicates_mapping.csv"

contacts_mapping = pd.read_csv(contacts_mapping_path, dtype={"old_contact_id": "string", "master_contact_id": "string"})

deals["contact_id"] = deals["contact_id"].astype("string").str.strip()
contacts_mapping["old_contact_id"] = contacts_mapping["old_contact_id"].astype("string").str.strip()
contacts_mapping["master_contact_id"] = contacts_mapping["master_contact_id"].astype("string").str.strip()

contact_id_mapping = dict(zip(contacts_mapping["old_contact_id"], contacts_mapping["master_contact_id"]))

affected_deals = deals["contact_id"].isin(contact_id_mapping.keys()).sum()

deals["contact_id"] = deals["contact_id"].replace(contact_id_mapping)

print("Перепривязано сделок:", affected_deals)

Перепривязано сделок: 35


# Работаем с дубликатами

In [446]:
# Полные дубликаты по всем столбцам
full_duplicates_count = deals.duplicated().sum()
full_duplicates_count

np.int64(0)

In [447]:
# Дубликаты по id
id_duplicates_count = deals["id"].duplicated().sum()
id_duplicates_count

np.int64(0)

### Проверяем бизнес-дубли

In [449]:
# Бизнес-дубль для Deals — это потенциально одна и та же сделка, созданная повторно в CRM. Проверяем по контакту, времени создания, стадии, продукту и суммам.
business_duplicate_subset = ["contact_id", "created_time", "stage", "product", "initial_amount_paid", "offer_total_amount"]
business_duplicates = deals[deals.duplicated(subset=business_duplicate_subset, keep=False)].sort_values(business_duplicate_subset)
business_duplicates.shape[0]

88

In [450]:
# Сколько строк было бы удалено, если оставить первую строку в каждой группе
deals.duplicated(subset=business_duplicate_subset, keep="first").sum()

np.int64(46)

### Проверяем дубли, отмеченные в Lost Reason

In [451]:
lost_reason_duplicates = deals[deals["lost_reason"].astype("string").str.lower().str.contains("duplicate|дубликат", na=False)]
lost_reason_duplicates.shape

(1771, 23)

In [452]:
lost_reason_duplicates[["id", "contact_id", "deal_owner_name", "stage", "lost_reason", "created_time", "closing_date", "product",
    "initial_amount_paid", "offer_total_amount", "months_of_study"]].head()

,id,contact_id,deal_owner_name,stage,lost_reason,created_time,closing_date,product,initial_amount_paid,offer_total_amount,months_of_study
8,5805028000056845137,5805028000056849237,Rachel White,Lost,Duplicate,21.06.2024 12:40,21.06.2024,NaN,NaN,NaN,NaN
10,5805028000056892253,5805028000056828292,Ulysses Adams,Lost,Duplicate,21.06.2024 12:32,21.06.2024,NaN,NaN,NaN,NaN
16,5805028000056828139,5805028000056824095,Rachel White,Lost,Duplicate,21.06.2024 11:44,21.06.2024,NaN,NaN,NaN,NaN
43,5805028000056705395,5805028000056727001,Ulysses Adams,Lost,Duplicate,20.06.2024 17:36,20.06.2024,NaN,NaN,NaN,NaN
44,5805028000056703432,5805028000056727001,Ulysses Adams,Lost,Duplicate,20.06.2024 17:34,20.06.2024,NaN,NaN,NaN,NaN


In [454]:
# Проверяем, есть ли среди lost_reason = Duplicate строки с признаками оплаты/обучения
duplicate_with_study_or_payment = lost_reason_duplicates[(lost_reason_duplicates["months_of_study"].notna())
    | (lost_reason_duplicates["initial_amount_paid"].notna()) | (lost_reason_duplicates["offer_total_amount"].notna())]

duplicate_with_study_or_payment.shape

(106, 23)

## Предварительный вывод по дубликатам
В Deals сначала удаляем только технически пустые строки без `id`.
* Полные дубли и дубли по `id` проверяются отдельно.  
* Бизнес-дубли и строки с `lost_reason = Duplicate` требуют осторожного анализа, потому что Deals — основная таблица для воронки, выручки и юнит-экономики.
* Строки, помеченные как дубликаты в `lost_reason`, вероятнее всего являются повторно созданными лидами. Но перед удалением проверяем, нет ли среди них строк с признаками обучения или оплаты.

In [455]:
# Удаляем строки, помеченные как дубликаты в Lost Reason
# Логика: если строка помечена как дубликат и при этом нет признаков учебы, ее можно удалить из анализа воронки, чтобы не завышать количество лидов.
# Создаем маску для строк, которые CRM пометила как дубликаты

lost_reason_duplicate_mask = (deals["lost_reason"].astype("string").str.lower().str.contains("duplicate|дубликат", na=False))

# Проверяем, есть ли среди этих строк студенты
lost_duplicates_check = deals[lost_reason_duplicate_mask].copy()
lost_duplicates_check_summary = pd.DataFrame({
    "metric": ["Строк с lost_reason = Duplicate","Из них с months_of_study","Из них со stage = Payment Done","Из них с initial_amount_paid","Из них с offer_total_amount"],
    "value": [len(lost_duplicates_check), lost_duplicates_check["months_of_study"].notna().sum(), (lost_duplicates_check["stage"] == "Payment Done").sum(),
        lost_duplicates_check["initial_amount_paid"].notna().sum(), lost_duplicates_check["offer_total_amount"].notna().sum()]})

lost_duplicates_check_summary

,metric,value
0,Строк с lost_reason = Duplicate,1771
1,Из них с months_of_study,2
2,Из них со stage = Payment Done,2
3,Из них с initial_amount_paid,105
4,Из них с offer_total_amount,105


In [456]:
# Удаляем только те Duplicate, где нет факта обучения.
# Если months_of_study заполнен, строку не трогаем.

duplicates_to_drop_mask = (lost_reason_duplicate_mask & deals["months_of_study"].isna())
duplicates_to_drop_count = duplicates_to_drop_mask.sum()
duplicates_to_drop_count

np.int64(1769)

In [457]:
# Удаляем такие строки
n_before_lost_duplicates = len(deals)
deals = deals[~duplicates_to_drop_mask].reset_index(drop=True)

print("Строк до удаления lost_reason Duplicate:", n_before_lost_duplicates)
print("Строк после удаления:", len(deals))
print("Удалено строк:", n_before_lost_duplicates - len(deals))

Строк до удаления lost_reason Duplicate: 21593
Строк после удаления: 19824
Удалено строк: 1769


### Вывод по Lost Reason = Duplicate

В `lost_reason` есть значения, указывающие на дубликат сделки. Согласно пояснениям к проекту, это означает, что лид был создан повторно, а одна из карточек была закрыта как дубль.

Такие строки удаляем только при отсутствии признака обучения (`months_of_study` пустой). Если `months_of_study` заполнен, строку не удаляем, так как это может быть фактический студент и его нельзя потерять из анализа выручки и юнит-экономики.

# Работаем с датами

In [460]:
date_columns = ["created_time", "closing_date"]
for col in date_columns:
    deals[col] = pd.to_datetime(deals[col], dayfirst=True, errors="coerce")
deals[date_columns].dtypes

created_time    datetime64[ns]
closing_date    datetime64[ns]
dtype: object

In [461]:
# Проверяем даты после преобразования
for col in date_columns:
    print(f"{col}:")
    print("  пустых дат:", deals[col].isna().sum())
    print("  минимальная дата:", deals[col].min())
    print("  максимальная дата:", deals[col].max())

created_time:
  пустых дат: 0
  минимальная дата: 2023-07-03 17:03:00
  максимальная дата: 2024-06-21 15:30:00
closing_date:
  пустых дат: 6686
  минимальная дата: 2022-10-11 00:00:00
  максимальная дата: 2024-12-11 00:00:00


In [462]:
# Проверяем ошибки: дата закрытия раньше даты создания
wrong_closing_date = deals[deals["closing_date"].notna() & deals["created_time"].notna() & (deals["closing_date"] < deals["created_time"].dt.normalize())]
wrong_closing_date.shape

(38, 23)

In [463]:
wrong_closing_date[["id", "contact_id", "stage", "lost_reason", "created_time", "closing_date"]].head()

,id,contact_id,stage,lost_reason,created_time,closing_date
427,5805028000055502890,5805028000055488584,Lost,Changed Decision,2024-06-16 00:06:00,2024-06-11
1961,5805028000051847114,5805028000051866233,Lost,Changed Decision,2024-05-25 21:29:00,2024-05-22
2627,5805028000049539444,5805028000049438717,Lost,Changed Decision,2024-05-12 11:19:00,2024-05-07
2853,5805028000048886321,5805028000048886280,Payment Done,NaN,2024-05-08 15:31:00,2024-05-07
2863,5805028000048886160,5805028000048886140,Payment Done,NaN,2024-05-08 12:54:00,2024-05-07


### Исправляем ошибки в closing_date

In [464]:
# Closing_date раньше created_time. Для потерянных сделок Lost заменяем closing_date на дату создания сделки.
# Это консервативное решение: сделка была закрыта как потерянная, но дата закрытия в CRM записана некорректно.
wrong_lost_closing_date_mask = (
    deals["closing_date"].notna()
    & deals["created_time"].notna()
    & (deals["closing_date"] < deals["created_time"].dt.normalize())
    & (deals["stage"] == "Lost"))
wrong_lost_closing_date_count = wrong_lost_closing_date_mask.sum()
wrong_lost_closing_date_count

np.int64(34)

In [465]:
deals.loc[wrong_lost_closing_date_mask, "closing_date"] = deals.loc[wrong_lost_closing_date_mask, "created_time"].dt.normalize()

In [468]:
# Проверяем, остались ли ошибки дат
wrong_closing_date_after = deals[
    deals["closing_date"].notna()
    & deals["created_time"].notna()
    & (deals["closing_date"] < deals["created_time"].dt.normalize())]

wrong_closing_date_after.shape

(4, 23)

In [469]:
wrong_closing_date_after[["id", "contact_id", "stage", "lost_reason", "created_time", "closing_date"]].head()

,id,contact_id,stage,lost_reason,created_time,closing_date
2853,5805028000048886321,5805028000048886280,Payment Done,NaN,2024-05-08 15:31:00,2024-05-07
2863,5805028000048886160,5805028000048886140,Payment Done,NaN,2024-05-08 12:54:00,2024-05-07
5994,5805028000042015392,5805028000009072093,Payment Done,NaN,2024-04-05 10:40:00,2023-10-03
12964,5805028000022036007,<NA>,Payment Done,NaN,2023-12-19 10:37:00,2023-07-01


## Вывод по датам
* Столбцы `created_time` и `closing_date` были приведены к формату datetime.
* Были найдены строки, где `closing_date` раньше `created_time`. Для сделок со статусом `Lost` такие значения были исправлены: `closing_date` заменена на дату создания сделки. Это сделано потому, что сделка была закрыта как потерянная, но дата закрытия была записана в CRM некорректно.
* Если после этого остаются строки с некорректной датой закрытия по другим статусам, их не исправляем автоматически, а оставляем как аномалии для контроля качества данных.

## Работаем с SLA

In [470]:
# Проверяем типы значений в SLA
deals["sla"].apply(type).value_counts()

sla
<class 'datetime.time'>         13238
<class 'float'>                  4827
<class 'datetime.timedelta'>     1759
Name: count, dtype: int64

In [471]:
# Проверяем пропуски в SLA
sla_missing = deals["sla"].isna().sum()
sla_missing_pct = round(deals["sla"].isna().mean() * 100, 2)
print("Пропусков в SLA:", sla_missing)
print("Доля пропусков в SLA:", sla_missing_pct, "%")

Пропусков в SLA: 4827
Доля пропусков в SLA: 24.35 %


In [472]:
# Функция для перевода SLA в минуты
def sla_to_minutes(value):
    if pd.isna(value):
        return np.nan
    
    # Если значение уже timedelta
    if isinstance(value, datetime.timedelta):
        return round(value.total_seconds() / 60, 2)
    
    # Если значение является временем
    if isinstance(value, datetime.time):
        return round(value.hour * 60 + value.minute + value.second / 60, 2)
    
    # Если значение является timestamp
    if isinstance(value, pd.Timestamp):
        return round(value.hour * 60 + value.minute + value.second / 60, 2)
    
    # Если значение числовое, предполагаем, что это секунды
    if isinstance(value, (int, float, np.integer, np.floating)):
        return round(value / 60, 2)
    
    # Если значение строковое вида HH:MM:SS или MM:SS
    value = str(value).strip()
    
    try:
        parts = value.split(":")
        
        if len(parts) == 3:
            hours = int(parts[0])
            minutes = int(parts[1])
            seconds = int(parts[2])
            return round(hours * 60 + minutes + seconds / 60, 2)
        
        if len(parts) == 2:
            minutes = int(parts[0])
            seconds = int(parts[1])
            return round(minutes + seconds / 60, 2)
    
    except:
        return np.nan
    
    return np.nan

In [475]:
# Создаем числовой столбец SLA в минутах
deals["sla_minutes"] = deals["sla"].apply(sla_to_minutes)
deals[["sla", "sla_minutes"]].head()

,sla,sla_minutes
0,NaN,NaN
1,NaN,NaN
2,00:26:43,26.72
3,01:00:04,60.07
4,00:53:12,53.20


In [476]:
# проверяем, остались ли значения, которые не распарсились
sla_not_parsed = deals[deals["sla"].notna() & deals["sla_minutes"].isna()]
sla_not_parsed.shape

(0, 24)

In [477]:
# Проверяем результат преобразования
print("Пропусков в исходном SLA:", deals["sla"].isna().sum())
print("Пропусков в sla_minutes:", deals["sla_minutes"].isna().sum())
deals["sla_minutes"].describe()

Пропусков в исходном SLA: 4827
Пропусков в sla_minutes: 4827


count     14997.000000
mean       1857.928822
std       12107.872569
min           0.050000
25%          73.330000
50%         329.850000
75%         932.200000
max      448474.400000
Name: sla_minutes, dtype: float64

In [478]:
# создадим флаг пропусков, для дальнейшей обработки
deals["sla_was_missing"] = deals["sla"].isna()
deals["sla_was_missing"].value_counts(dropna=False)

sla_was_missing
False    14997
True      4827
Name: count, dtype: int64

In [479]:
if "sla_minutes" in deals.columns:
    deals["sla"] = deals["sla_minutes"]

In [480]:
deals.head()

,id,deal_owner_name,closing_date,quality,stage,lost_reason,page,campaign,sla,content,...,created_time,course_duration,months_of_study,initial_amount_paid,offer_total_amount,contact_id,city,level_of_deutsch,sla_minutes,sla_was_missing
0,5805028000056864695,Ben Hall,NaT,NaN,New Lead,NaN,/eng/test,03.07.23women,NaN,v16,...,2024-06-21 15:30:00,NaN,NaN,NaN,NaN,5805028000056849495,NaN,NaN,NaN,True
1,5805028000056859489,Ulysses Adams,NaT,NaN,New Lead,NaN,/at-eng,NaN,NaN,NaN,...,2024-06-21 15:23:00,6.0,NaN,0,2000,5805028000056834471,NaN,NaN,NaN,True
2,5805028000056832357,Ulysses Adams,2024-06-21,D - Non Target,Lost,Non target,/at-eng,engwien_AT,26.72,b1-at,...,2024-06-21 14:45:00,NaN,NaN,NaN,NaN,5805028000056854421,NaN,NaN,26.72,False
3,5805028000056824246,Eva Kent,2024-06-21,E - Non Qualified,Lost,Invalid number,/eng,04.07.23recentlymoved_DE,60.07,bloggersvideo14com,...,2024-06-21 13:32:00,NaN,NaN,NaN,NaN,5805028000056889351,NaN,NaN,60.07,False
4,5805028000056873292,Ben Hall,2024-06-21,D - Non Target,Lost,Non target,/eng,discovery_DE,53.20,website,...,2024-06-21 13:21:00,NaN,NaN,NaN,NaN,5805028000056876176,NaN,NaN,53.20,False


## Работа со столбцом SLA
* `SLA` показывает время ответа менеджера на заявку. Значения SLA были представлены в разных технических форматах, включая строки времени и `timedelta`. Поэтому сначала была написана функция для универсального преобразования SLA в минуты.
* Все непустые значения SLA были преобразованы в новый числовой столбец `sla_minutes`.
* Пропуски в SLA не заполнялись медианой, потому что SLA — это фактическое время реакции менеджера. Искусственное заполнение могло бы исказить анализ скорости обработки лидов. Вместо этого создан флаг `sla_was_missing`.

## Обработка денежных полей

In [481]:
money_columns = ["initial_amount_paid", "offer_total_amount"]
deals[money_columns].head()

,initial_amount_paid,offer_total_amount
0,NaN,NaN
1,0,2000
2,NaN,NaN
3,NaN,NaN
4,NaN,NaN


In [482]:
# Проверяем типы данных
deals[money_columns].dtypes

initial_amount_paid    object
offer_total_amount     object
dtype: object

In [483]:
# Проверяем пропуски
deals[money_columns].isna().sum()

initial_amount_paid    15762
offer_total_amount     15742
dtype: int64

In [484]:
# Смотрим уникальные значения, чтобы понять, есть ли грязные символы
for col in money_columns:
    print(f"\n{col}")
    print(deals[col].astype("string").dropna().unique()[:30])


initial_amount_paid
<StringArray>
[         '0',       '1000', '€ 3.500,00',        '500',        '100',
       '4500',        '300',        '200',       '2000',      '11000',
       '4000',       '3000',       '3500',      '11500',       '1200',
       '1500',       '5000',          '1',        '600',        '700',
        '350',          '9',        '400',        '450']
Length: 24, dtype: string

offer_total_amount
<StringArray>
[      '2000',       '9000',      '11000',       '3500',       '4500',
 '€ 2.900,00',       '6500',       '4000',       '3000',      '10000',
       '2500',       '5000',      '11500',          '1',       '1000',
       '1200',          '0',       '1500', '€ 11398,00',      '11111',
       '6000']
Length: 21, dtype: string


### Очищаем денежные поля

In [485]:
def clean_money(value):
    if pd.isna(value):
        return np.nan
    
    value = str(value).strip()
    
    # Убираем валюты, пробелы и лишние символы
    value = (value.replace("€", "").replace("$", "").replace("ˆ", "").replace(" ", ""))
    
    # Если есть европейский формат 3.500,00 — приводим к 3500.00
    if "," in value:
        value = value.replace(".", "")
        value = value.replace(",", ".")
    
    return pd.to_numeric(value, errors="coerce")

In [486]:
for col in money_columns:
    deals[col] = deals[col].apply(clean_money)
deals[money_columns].dtypes

initial_amount_paid    float64
offer_total_amount     float64
dtype: object

In [487]:
deals[money_columns].describe()

,initial_amount_paid,offer_total_amount
count,4062.000000,4082.000000
mean,957.117184,7209.997550
std,1415.375373,4597.443616
min,0.000000,0.000000
25%,300.000000,3500.000000
50%,1000.000000,11000.000000
75%,1000.000000,11000.000000
max,11500.000000,11500.000000


In [488]:
# Проверяем значения, которые не удалось привести к числу
for col in money_columns:
    bad_values = deals[deals[col].isna()]
    
    print(col, "пропусков после очистки:", bad_values.shape[0])

initial_amount_paid пропусков после очистки: 15762
offer_total_amount пропусков после очистки: 15742


### Проверяем логические ошибки в оплатах

In [489]:
# Initial Amount Paid больше Offer Total Amount
initial_more_than_total = deals[deals["initial_amount_paid"].notna() & deals["offer_total_amount"].notna() & (deals["initial_amount_paid"] > deals["offer_total_amount"])]
initial_more_than_total.shape

(55, 25)

In [490]:
initial_more_than_total[["id","stage","product","education_type","course_duration","months_of_study","initial_amount_paid","offer_total_amount"]].head()

,id,stage,product,education_type,course_duration,months_of_study,initial_amount_paid,offer_total_amount
1196,5805028000053717506,Call Delayed,Web Developer,Morning,6.0,NaN,3000.0,2900.0
1299,5805028000053561185,Lost,UX/UI Design,Morning,11.0,NaN,11500.0,11000.0
1313,5805028000053462041,Lost,UX/UI Design,Morning,11.0,NaN,11500.0,11000.0
1344,5805028000053242748,Waiting For Payment,UX/UI Design,Morning,11.0,NaN,11500.0,11000.0
1355,5805028000053242571,Lost,UX/UI Design,Morning,11.0,NaN,11500.0,11000.0


### Предварительный вывод по денежным полям
* Денежные поля `initial_amount_paid` и `offer_total_amount` были очищены от валютных символов, пробелов и нестандартных символов, после чего приведены к числовому формату.
* Отдельно проверили логические ошибки: например, случаи, где `initial_amount_paid` больше `offer_total_amount`. Такие строки нельзя исправлять автоматически без анализа, потому что в данных могут быть как ошибки ввода, так и перестановка значений между столбцами.

### Создаем ключевые флаги для оплаты и обучения

In [494]:
# Флаг: сделка имеет статус Payment Done
deals["is_payment_done_stage"] = deals["stage"].eq("Payment Done")

In [495]:
# Флаг: человек реально учится по условиям проекта Months of study — главный признак того, что студент начал обучение.
deals["is_student"] = deals["months_of_study"].notna()

In [496]:
# Флаг: выручку можно учитывать, деньги учитываем только если человек учится и есть сумма первого платежа.
deals["is_revenue_valid"] = (deals["is_student"] & deals["initial_amount_paid"].notna())

In [497]:
# Проверяем пересечение статуса Payment Done и факта обучения
payment_student_summary = pd.crosstab(deals["is_payment_done_stage"],deals["is_student"], margins=True)
payment_student_summary

is_student,False,True,All
is_payment_done_stage,,,
False,18966,0,18966
True,18,840,858
All,18984,840,19824


In [498]:
# Проверяем строки, где Payment Done, но нет Months of study
payment_done_without_study = deals[deals["is_payment_done_stage"] & ~deals["is_student"]]
payment_done_without_study.shape

(18, 28)

In [499]:
payment_done_without_study[["id","stage","product","education_type","months_of_study","initial_amount_paid","offer_total_amount","closing_date"]].head()

,id,stage,product,education_type,months_of_study,initial_amount_paid,offer_total_amount,closing_date
5751,5805028000042448604,Payment Done,Find yourself in IT,NaN,NaN,1.0,1.0,2024-05-11
12964,5805028000022036007,Payment Done,NaN,NaN,NaN,0.0,0.0,2023-07-01
14070,5805028000019345087,Payment Done,NaN,NaN,NaN,0.0,0.0,2024-05-01
15038,5805028000017534101,Payment Done,NaN,NaN,NaN,NaN,NaN,NaT
15922,5805028000014229195,Payment Done,NaN,NaN,NaN,NaN,NaN,NaT


### Решение по buyer и выручке
*  Для проекта важно разделять факт оплаты в CRM и факт обучения.
`stage = Payment Done` означает, что по CRM деньги были получены.  
Но для расчета buyer и выручки используем `months_of_study`, потому что по пояснениям к проекту, если месяц обучения не заполнен, студент не учится и деньги учитывать нельзя (могут вернуть либо ошибка заполнения).

Поэтому:
- `is_payment_done_stage` показывает статус оплаты в CRM;
- `is_student` показывает факт обучения;
- `is_revenue_valid` показывает строки, где можно учитывать выручку.

### Проверяем денежные ошибки у студентов (именно у тех кто учится)

In [500]:
# Смотрим студентов с заполненными двумя денежными полями
students_with_money = deals[deals["is_student"] & deals["initial_amount_paid"].notna() & deals["offer_total_amount"].notna()].copy()
students_with_money.shape

(840, 28)

In [501]:
# Ошибка: первый платеж больше полной суммы предложения
students_initial_more_than_total = students_with_money[students_with_money["initial_amount_paid"] > students_with_money["offer_total_amount"]]
students_initial_more_than_total.shape

(21, 28)

### Исправляем очевидную перестановку Initial и Total (так как мы знаем что данные намеренно испорчены)


In [502]:
# Если студент учится, оба поля заполнены, initial_amount_paid > offer_total_amount, и offer_total_amount выглядит как возможный первый платеж,
# считаем, что значения были перепутаны местами.
swap_money_mask = (deals["is_student"] & deals["initial_amount_paid"].notna() & deals["offer_total_amount"].notna()
    & (deals["initial_amount_paid"] > deals["offer_total_amount"]))

swap_money_count = swap_money_mask.sum()
swap_money_count

np.int64(21)

In [503]:
# Меняем значения местами
deals.loc[swap_money_mask, ["initial_amount_paid", "offer_total_amount"]] = deals.loc[swap_money_mask, ["offer_total_amount", "initial_amount_paid"]].values

In [504]:
# Проверяем, остались ли такие ошибки у студентов
students_initial_more_than_total_after = deals[deals["is_student"] & deals["initial_amount_paid"].notna()
    & deals["offer_total_amount"].notna() & (deals["initial_amount_paid"] > deals["offer_total_amount"])]
students_initial_more_than_total_after.shape

(0, 28)

## Вывод по денежным ошибкам

* Для студентов (`months_of_study` заполнен) были проверены случаи, где `initial_amount_paid` больше `offer_total_amount`.
Такое соотношение является очевидной ошибкой, так как первый платеж не должен быть больше полной суммы предложения. Для таких строк значения `initial_amount_paid` и `offer_total_amount` были поменяны местами.

* Для строк без признака обучения деньги не исправлялись и не учитывались в выручке.

* Для денежных полей в Deals я не заполнял пропуски нулем массово. потому что NaN = сумма неизвестна / не зафиксирована, а 0 = сумма равна нулю. 
  * По FAQ проекта прямо сказано: если Payment Done, но сумма пустая — оплата была, просто сумма не зафиксирована;
если months_of_study пустой — студент не учится, деньги учитывать нельзя;
значения 0, 1, 9 могут быть демо-доступами или символическими оплатами.
  * Если мы заменим все пропуски на 0, то потеряем смысл.
  * И потом:
средний платеж занизится;
выручка занизится;
будет непонятно, где реально ноль, а где пропуск;
нельзя будет отдельно анализировать ошибки фиксации платежей.

In [505]:
# После swap нужно пересчитать is_revenue_valid, потому что денежные поля изменились
deals["is_revenue_valid"] = (deals["is_student"] & deals["initial_amount_paid"].notna())

In [506]:
# Итоговая сумма валидной выручки по первому платежу
valid_revenue = deals.loc[deals["is_revenue_valid"], "initial_amount_paid"].sum()
valid_revenue

np.float64(950350.0)

# Изучаем продуктовые и учебные поля

In [507]:
product_columns = ["payment_type", "product", "education_type", "course_duration", "months_of_study"]

product_info = pd.DataFrame({
    "column": product_columns,
    "non_null": deals[product_columns].notna().sum().values,
    "missing": deals[product_columns].isna().sum().values,
    "missing_pct": (deals[product_columns].isna().mean().values * 100).round(2),
    "unique_values": deals[product_columns].nunique(dropna=True).values})

product_info

,column,non_null,missing,missing_pct,unique_values
0,payment_type,484,19340,97.56,3
1,product,3539,16285,82.15,5
2,education_type,3258,16566,83.57,2
3,course_duration,3535,16289,82.17,2
4,months_of_study,840,18984,95.76,12


In [508]:
# Смотрим значения категориальных продуктовых полей
for col in ["payment_type", "product", "education_type"]:
    print(f"\n{col}")
    print("-" * 50)
    print(deals[col].value_counts(dropna=False).head())


payment_type
--------------------------------------------------
payment_type
NaN                   19340
Recurring Payments      342
One Payment             137
Reservation               5
Name: count, dtype: int64

product
--------------------------------------------------
product
NaN                    16285
Digital Marketing       1953
UX/UI Design            1015
Web Developer            567
Find yourself in IT        3
Name: count, dtype: int64

education_type
--------------------------------------------------
education_type
NaN        16566
Morning     2859
Evening      399
Name: count, dtype: int64


In [509]:
# Смотрим длительность курса и месяцы обучения
for col in ["course_duration", "months_of_study"]:
    print(f"\n{col}")
    print("-" * 50)
    print(deals[col].value_counts(dropna=False).sort_index())


course_duration
--------------------------------------------------
course_duration
6.0       567
11.0     2968
NaN     16289
Name: count, dtype: int64

months_of_study
--------------------------------------------------
months_of_study
0.0         1
1.0        67
2.0       104
3.0        94
4.0        93
5.0        64
6.0       107
7.0        79
8.0        83
9.0        61
10.0       42
11.0       45
NaN     18984
Name: count, dtype: int64


### Проверяем продуктовые поля у студентов

In [511]:
students = deals[deals["is_student"]].copy()
students[product_columns].isna().sum() # пропуски для студентов

payment_type       479
product              0
education_type       7
course_duration      0
months_of_study      0
dtype: int64

In [512]:
# Студенты с пропусками в важных продуктовых полях
students_missing_product_info = students[students["product"].isna() | students["education_type"].isna() | students["course_duration"].isna()]
students_missing_product_info[["id","contact_id","stage","product","education_type","course_duration","months_of_study","initial_amount_paid","offer_total_amount"]].head()

,id,contact_id,stage,product,education_type,course_duration,months_of_study,initial_amount_paid,offer_total_amount
15956,5805028000014077590,5805028000014115637,Payment Done,Digital Marketing,NaN,11.0,9.0,1000.0,11500.0
16012,5805028000013945394,5805028000013945362,Payment Done,Digital Marketing,NaN,11.0,9.0,1000.0,11500.0
16593,5805028000011780001,5805028000011759095,Payment Done,Digital Marketing,NaN,11.0,9.0,1000.0,11500.0
18028,5805028000007358035,5805028000007327089,Payment Done,Digital Marketing,NaN,11.0,10.0,1000.0,11500.0
18058,5805028000007093813,5805028000007082779,Payment Done,Digital Marketing,NaN,11.0,10.0,1000.0,11500.0


### Проверяем связи между продуктом, длительностью и форматом обучения

In [513]:
# Сводка по продукту, длительности курса и формату обучения
product_structure = (deals[deals["product"].notna()].groupby(["product", "course_duration", "education_type"], dropna=False)
    .agg(rows_count=("id", "count"), students_count=("is_student", "sum"), avg_total_amount=("offer_total_amount", "mean"), median_total_amount=("offer_total_amount", "median")).reset_index()
    .sort_values(["product", "course_duration", "education_type"]))

product_structure

,product,course_duration,education_type,rows_count,students_count,avg_total_amount,median_total_amount
0,Data Analytics,NaN,NaN,1,0,6000.000000,6000.0
1,Digital Marketing,11.0,Evening,246,113,3675.000000,4000.0
2,Digital Marketing,11.0,Morning,1509,354,10545.546622,11000.0
3,Digital Marketing,11.0,NaN,198,7,10939.655172,11500.0
4,Find yourself in IT,NaN,NaN,3,0,0.333333,0.0
5,UX/UI Design,11.0,Evening,152,58,3932.330827,4000.0
6,UX/UI Design,11.0,Morning,803,171,10335.107731,11000.0
7,UX/UI Design,11.0,NaN,60,0,10642.857143,11000.0
8,Web Developer,6.0,Evening,1,0,2000.000000,2000.0
9,Web Developer,6.0,Morning,538,137,5368.301887,5000.0


## Предварительный вывод по продуктовым полям

Поля `payment_type`, `product`, `education_type`, `course_duration` заполнены в основном для сделок, которые дошли до оплаты или обучения.

`months_of_study` не заполняем искусственно, потому что это ключевой признак фактического обучения.  
Для студентов (`months_of_study` заполнен) отдельно проверяем, есть ли пропуски в продукте, формате обучения и длительности курса. Если связь между продуктом, длительностью и форматом очевидна, такие пропуски можно будет дозаполнить точечно.

### Попытка дозаполнить продуктовые поля по истории контакта

In [514]:
# Поля, которые можно попробовать дозаполнить по другим сделкам того же contact_id
columns_to_backfill_by_contact = ["payment_type", "product", "education_type", "course_duration"]

In [515]:
# Сохраняем флаги, где были пропуски до дозаполнения
for col in columns_to_backfill_by_contact:
    deals[f"{col}_was_missing"] = deals[col].isna()

In [516]:
# Дозаполняем только внутри одного contact_id. Берем ближайшие известные значения по этому же контакту.
deals = deals.sort_values(["contact_id", "created_time"]).reset_index(drop=True)
known_contact_mask = deals["contact_id"].notna()

for col in columns_to_backfill_by_contact:
    deals.loc[known_contact_mask, col] = (deals.loc[known_contact_mask].groupby("contact_id")[col].transform(lambda x: x.ffill().bfill()))

C:\Users\Admin\AppData\Local\Temp\ipykernel_41024\1344904609.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  deals.loc[known_contact_mask, col] = (deals.loc[known_contact_mask].groupby("contact_id")[col].transform(lambda x: x.ffill().bfill()))
C:\Users\Admin\AppData\Local\Temp\ipykernel_41024\1344904609.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  deals.loc[known_contact_mask, col] = (deals.loc[known_contact_mask].groupby("contact_id")[col].transform(lambda x: x.ffill().bfill()))
C:\Users\Admin\AppData\Local\Temp\ipykernel_41024\1344904609.py:

In [517]:
# Проверяем, сколько пропусков было заполнено
backfill_summary = []

for col in columns_to_backfill_by_contact:
    missing_before = deals[f"{col}_was_missing"].sum()
    missing_after = deals[col].isna().sum()
    
    backfill_summary.append({
        "column": col,
        "missing_before": missing_before,
        "missing_after": missing_after,
        "filled": missing_before - missing_after})

backfill_summary = pd.DataFrame(backfill_summary)

backfill_summary

,column,missing_before,missing_after,filled
0,payment_type,19340,19176,164
1,product,16285,15430,855
2,education_type,16566,15749,817
3,course_duration,16289,15434,855


### Важное ограничение backfill
* Дозаполнение выполняется только по `contact_id`, то есть значение переносится между сделками одного и того же контакта.
* Такой подход не создает данные из воздуха, а использует уже известную информацию по тому же клиенту. При этом `months_of_study` не заполняется, потому что это главный признак факта обучения.

In [519]:
# Теперь отдельно проверяем студентов после дозаполнения:
students_after_backfill = deals[deals["is_student"]].copy()
students_after_backfill[columns_to_backfill_by_contact].isna().sum()

payment_type       476
product              0
education_type       7
course_duration      0
dtype: int64

In [520]:
students_after_backfill[students_after_backfill["product"].isna() | students_after_backfill["education_type"].isna()
    | students_after_backfill["course_duration"].isna()][[
    "id",
    "contact_id",
    "stage",
    "product",
    "education_type",
    "course_duration",
    "months_of_study",
    "initial_amount_paid",
    "offer_total_amount"]]

,id,contact_id,stage,product,education_type,course_duration,months_of_study,initial_amount_paid,offer_total_amount
61,5805028000001401001,5805028000001350049,Payment Done,Digital Marketing,NaN,11.0,8.0,1000.0,11500.0
1229,5805028000004313387,5805028000004333294,Payment Done,Digital Marketing,NaN,11.0,11.0,1000.0,11000.0
2182,5805028000007093813,5805028000007082779,Payment Done,Digital Marketing,NaN,11.0,10.0,1000.0,11500.0
2266,5805028000007358035,5805028000007327089,Payment Done,Digital Marketing,NaN,11.0,10.0,1000.0,11500.0
3940,5805028000011780001,5805028000011759095,Payment Done,Digital Marketing,NaN,11.0,9.0,1000.0,11500.0
4588,5805028000013945394,5805028000013945362,Payment Done,Digital Marketing,NaN,11.0,9.0,1000.0,11500.0
4690,5805028000014077590,5805028000014115637,Payment Done,Digital Marketing,NaN,11.0,9.0,1000.0,11500.0


### Заполняем оставшиеся пропуски Unknown

In [521]:
# Для категориальных полей оставшиеся пропуски заполняем Unknown. course_duration оставляем NaN, так как это числовое поле.
categorical_product_columns = ["payment_type", "product", "education_type"]
for col in categorical_product_columns:
    deals[col] = deals[col].fillna("Unknown")

In [522]:
deals[categorical_product_columns + ["course_duration"]].isna().sum()

payment_type           0
product                0
education_type         0
course_duration    15434
dtype: int64

## Вывод по продуктовым полям
* Продуктовые поля были дозаполнены по истории того же контакта (`contact_id`), если у клиента в другой сделке уже было известно значение.
* Оставшиеся пропуски в категориальных полях `payment_type`, `product`, `education_type` заполнены значением `Unknown`, чтобы эти строки не выпадали из группировок.  
* `course_duration` не заполняется значением `Unknown`, так как это числовое поле. Искусственно подставлять длительность курса без надежного основания нельзя.
* `months_of_study` не заполнялся, так как он используется как основной признак фактического обучения.

# Работаем с маркетинговыми полями

In [523]:
marketing_columns = ["page", "campaign", "content", "term", "source"]

marketing_info = pd.DataFrame({
    "column": marketing_columns,
    "non_null": deals[marketing_columns].notna().sum().values,
    "missing": deals[marketing_columns].isna().sum().values,
    "missing_pct": (deals[marketing_columns].isna().mean().values * 100).round(2),
    "unique_values": deals[marketing_columns].nunique(dropna=True).values})

marketing_info

,column,non_null,missing,missing_pct,unique_values
0,page,19824,0,0.00,33
1,campaign,15584,4240,21.39,153
2,content,13799,6025,30.39,184
3,term,12101,7723,38.96,217
4,source,19824,0,0.00,13


In [524]:
# Смотрим основные значения по source
deals["source"].value_counts(dropna=False)

source
Facebook Ads      4730
Google Ads        4115
Tiktok Ads        2003
SMM               1669
Youtube Ads       1618
Organic           1499
CRM               1456
Bloggers          1074
Telegram posts     993
Webinar            306
Partnership        203
Test               156
Offline              2
Name: count, dtype: int64

In [525]:
# Смотрим основные значения по page
deals["page"].value_counts(dropna=False).head(10)

page
/eng                     5682
eng/digital-marketing    4371
/eng/test                2766
/webinar                 1115
/workshop                1059
/direct                  1040
/eng/ux-ui               1020
/web-developer            638
/pl-eng                   439
/email                    396
Name: count, dtype: int64

### Дозаполняем маркетинговые поля по истории контакта

In [526]:
# Маркетинговые поля пробуем дозаполнить только по тому же contact_id. Campaign не разбираем по датам внутри названия — по условиям проекта это внутреннее название кампании.
marketing_columns_to_backfill = ["page", "campaign", "content", "term", "source"]

for col in marketing_columns_to_backfill:
    deals[f"{col}_was_missing"] = deals[col].isna()

In [527]:
known_contact_mask = deals["contact_id"].notna()
deals = deals.sort_values(["contact_id", "created_time"]).reset_index(drop=True)
for col in marketing_columns_to_backfill:
    deals.loc[known_contact_mask, col] = (deals.loc[known_contact_mask].groupby("contact_id")[col].transform(lambda x: x.ffill().bfill())    )

C:\Users\Admin\AppData\Local\Temp\ipykernel_41024\3028840048.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  deals.loc[known_contact_mask, col] = (deals.loc[known_contact_mask].groupby("contact_id")[col].transform(lambda x: x.ffill().bfill())    )
C:\Users\Admin\AppData\Local\Temp\ipykernel_41024\3028840048.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  deals.loc[known_contact_mask, col] = (deals.loc[known_contact_mask].groupby("contact_id")[col].transform(lambda x: x.ffill().bfill())    )
C:\Users\Admin\AppData\Local\Temp\ipykernel_41024\302884

In [528]:
# Проверяем, сколько пропусков удалось заполнить
marketing_backfill_summary = []

for col in marketing_columns_to_backfill:
    missing_before = deals[f"{col}_was_missing"].sum()
    missing_after = deals[col].isna().sum()
    
    marketing_backfill_summary.append({
        "column": col,
        "missing_before": missing_before,
        "missing_after": missing_after,
        "filled": missing_before - missing_after})

marketing_backfill_summary = pd.DataFrame(marketing_backfill_summary)

marketing_backfill_summary

,column,missing_before,missing_after,filled
0,page,0,0,0
1,campaign,4240,3417,823
2,content,6025,4720,1305
3,term,7723,6640,1083
4,source,0,0,0


### Оставшиеся пропуски заполняем Unknown

In [529]:
for col in marketing_columns_to_backfill:
    deals[col] = deals[col].fillna("Unknown")

In [530]:
deals[marketing_columns_to_backfill].isna().sum()

page        0
campaign    0
content     0
term        0
source      0
dtype: int64

## Вывод по маркетинговым полям
* Маркетинговые поля `page`, `campaign`, `content`, `term`, `source` используются для анализа источников лидов и эффективности рекламных кампаний.
* Сначала была проверена возможность восстановить пропуски по истории того же контакта (`contact_id`). Если у клиента в другой сделке уже было известно значение маркетингового поля, оно переносилось на пустые строки этого же клиента.
* Оставшиеся пропуски были заполнены значением `Unknown`, чтобы строки не выпадали из группировок.
* Названия кампаний не разбирались по датам внутри строки, так как по условиям проекта `Campaign` — это внутреннее название, которое задают маркетологи, и единого правила формирования нет.

# Работаем с городами

In [531]:
# Смотрим пропуски и количество уникальных значений
city_info = pd.DataFrame({
    "column": ["city"],
    "non_null": [deals["city"].notna().sum()],
    "missing": [deals["city"].isna().sum()],
    "missing_pct": [round(deals["city"].isna().mean() * 100, 2)],
    "unique_values": [deals["city"].nunique(dropna=True)]})

city_info

,column,non_null,missing,missing_pct,unique_values
0,city,2510,17314,87.34,876


In [532]:
# Смотрим самые частые значения
deals["city"].value_counts(dropna=False).head(20)

city
NaN                       17314
-                           348
Berlin                      182
München                      74
Hamburg                      62
Nürnberg                     45
Leipzig                      45
Düsseldorf                   33
Dresden                      28
Frankfurt                    27
Dortmund                     26
Köln                         25
Stuttgart                    20
Hannover                     19
Duisburg                     19
Bremen                       17
Karlsruhe                    16
Essen                        16
Bochum                       15
Villingen-Schwenningen       14
Name: count, dtype: int64

### Нормализуем город

In [533]:
# Приводим к строке, убираем лишние пробелы
deals["city"] = deals["city"].astype("string").str.strip()

In [534]:
# Если в city попала улица или лишняя информация через запятую, оставляем первую часть до запятой.
deals["city"] = deals["city"].str.split(",").str[0].str.strip()

In [535]:
# Если остались пустые строки, заменяем на NaN
deals["city"] = deals["city"].replace("", np.nan)

### Дозаполняем город по истории контакта

In [536]:
deals["city_was_missing"] = deals["city"].isna()
known_contact_mask = deals["contact_id"].notna()
deals = deals.sort_values(["contact_id", "created_time"]).reset_index(drop=True)

deals.loc[known_contact_mask, "city"] = (
    deals
    .loc[known_contact_mask]
    .groupby("contact_id")["city"]
    .transform(lambda x: x.ffill().bfill()))

In [537]:
city_missing_before = deals["city_was_missing"].sum()
city_missing_after = deals["city"].isna().sum()

print("Пропусков city до дозаполнения:", city_missing_before)
print("Пропусков city после дозаполнения:", city_missing_after)
print("Заполнено:", city_missing_before - city_missing_after)

Пропусков city до дозаполнения: 17314
Пропусков city после дозаполнения: 16690
Заполнено: 624


In [538]:
# Оставшиеся пропуски заполняем Unknown
deals["city"] = deals["city"].fillna("Unknown")

In [539]:
deals["city"].value_counts(dropna=False).head()

city
Unknown    16690
-            426
Berlin       264
München       88
Hamburg       78
Name: count, dtype: int64

## Вывод по городам
* Поле `city` было нормализовано: значения приведены к строковому формату, убраны лишние пробелы и дополнительная информация после запятой.
* Часть пропусков была дозаполнена по истории того же контакта (`contact_id`). Оставшиеся пропуски заполнены значением `Unknown`, чтобы строки не выпадали из географического анализа.

# Работаем с уровнем немецкого языка

In [540]:
# Смотрим пропуски и количество уникальных значений
deutsch_info = pd.DataFrame({
    "column": ["level_of_deutsch"],
    "non_null": [deals["level_of_deutsch"].notna().sum()],
    "missing": [deals["level_of_deutsch"].isna().sum()],
    "missing_pct": [round(deals["level_of_deutsch"].isna().mean() * 100, 2)],
    "unique_values": [deals["level_of_deutsch"].nunique(dropna=True)]})
deutsch_info

,column,non_null,missing,missing_pct,unique_values
0,level_of_deutsch,1247,18577,93.71,214


In [541]:
# Смотрим примеры значений
deals["level_of_deutsch"].value_counts(dropna=False).head(20)

level_of_deutsch
NaN      18577
B1         219
б1         118
в1         100
Б1          93
b1          93
В1          62
А2          53
B2          44
а2          32
б2          30
Б2          20
в2          20
b2          19
в1-в2       18
A2          18
?           15
В2          14
с1          13
А1          11
Name: count, dtype: int64

### Нормализуем уровень немецкого

In [542]:
def normalize_deutsch_level(value):
    if pd.isna(value):
        return "Unknown"
    
    value = str(value).strip().upper()
    
    # Заменяем похожие кириллические буквы на латинские
    replacements = {
        "А": "A",
        "В": "B",
        "С": "C",
        "Б": "B"
    }
    
    for old, new in replacements.items():
        value = value.replace(old, new)
    
    # Ищем стандартный уровень A0, A1, A2, B1, B2, C1, C2
    match = re.search(r"\b(A0|A1|A2|B1|B2|C1|C2)\b", value)
    
    if match:
        return match.group(1)
    
    return "Unknown"

In [543]:
deals["level_of_deutsch_clean"] = deals["level_of_deutsch"].apply(normalize_deutsch_level)
deals["level_of_deutsch_clean"].value_counts(dropna=False)

level_of_deutsch_clean
Unknown    18630
B1           814
B2           170
A2           149
C1            27
A1            25
A0             6
C2             3
Name: count, dtype: int64

In [544]:
if "level_of_deutsch_clean" in deals.columns:
    deals["level_of_deutsch"] = deals["level_of_deutsch_clean"]

## Вывод по уровню немецкого
* Поле `level_of_deutsch` содержит значения в разных форматах: латиница, кириллица, смешанный регистр и свободный текст.
* Для анализа был создан нормализованный столбец `level_of_deutsch_clean`, где значения приведены к стандартным уровням: `A0`, `A1`, `A2`, `B1`, `B2`, `C1`, `C2`.  
* Значения, которые не удалось надежно распознать, отнесены к категории `Unknown`.

## Ревизия финальных столбцов
* Перед сохранением проводим ревизию созданных технических полей. В финальный файл включаем только исходные аналитически значимые столбцы и один дополнительный признак is_student (важно для понимания баеров).

* Технические вспомогательные признаки, которые использовались только на этапе очистки, в финальный файл не включаем. Столбец sla был сохранен в числовом формате минут. Столбец level_of_deutsch был сохранен в нормализованном виде.

In [546]:
final_deals_columns = [
    "id",
    "deal_owner_name",
    "closing_date",
    "quality",
    "stage",
    "lost_reason",
    "page",
    "campaign",
    "sla",
    "content",
    "term",
    "source",
    "payment_type",
    "product",
    "education_type",
    "created_time",
    "course_duration",
    "months_of_study",
    "initial_amount_paid",
    "offer_total_amount",
    "contact_id",
    "city",
    "level_of_deutsch",
    "is_student"]

deals_clean = deals[final_deals_columns].copy()
deals_clean.shape

(19824, 24)

In [547]:
deals_clean.dtypes

id                     string[python]
deal_owner_name                object
closing_date           datetime64[ns]
quality                        object
stage                          object
lost_reason                    object
page                           object
campaign                       object
sla                           float64
content                        object
term                           object
source                         object
payment_type                   object
product                        object
education_type                 object
created_time           datetime64[ns]
course_duration               float64
months_of_study               float64
initial_amount_paid           float64
offer_total_amount            float64
contact_id             string[python]
city                           object
level_of_deutsch               object
is_student                       bool
dtype: object

In [548]:
deals_clean.head()

,id,deal_owner_name,closing_date,quality,stage,lost_reason,page,campaign,sla,content,...,education_type,created_time,course_duration,months_of_study,initial_amount_paid,offer_total_amount,contact_id,city,level_of_deutsch,is_student
0,5805028000005176025,John Doe,NaT,NaN,Registered on Webinar,NaN,/workshop,web2408_DE,NaN,Unknown,...,Morning,2023-08-18 19:42:00,11.0,NaN,NaN,NaN,5805028000000872003,Unknown,Unknown,False
1,5805028000005168037,John Doe,NaT,NaN,Registered on Webinar,NaN,/workshop,web2408_DE,NaN,Unknown,...,Morning,2023-08-18 19:45:00,11.0,NaN,NaN,NaN,5805028000000872003,Unknown,Unknown,False
2,5805028000005174051,John Doe,NaT,NaN,Registered on Webinar,NaN,/workshop,web2408_DE,NaN,Unknown,...,Morning,2023-08-18 19:54:00,11.0,NaN,NaN,NaN,5805028000000872003,Unknown,Unknown,False
3,5805028000005180061,John Doe,NaT,E - Non Qualified,Registered on Webinar,NaN,/workshop,web2408_DE,5422.58,Unknown,...,Morning,2023-08-18 19:58:00,11.0,NaN,100.0,4000.0,5805028000000872003,Unknown,Unknown,False
4,5805028000005176076,John Doe,NaT,NaN,Registered on Webinar,NaN,/workshop,web2408_DE,NaN,Unknown,...,Evening,2023-08-18 20:22:00,11.0,NaN,NaN,NaN,5805028000000872003,Unknown,Unknown,False


In [549]:
deals_clean.isna().sum()

id                         0
deal_owner_name           29
closing_date            6686
quality                 2236
stage                      0
lost_reason             5469
page                       0
campaign                   0
sla                     4827
content                    0
term                       0
source                     0
payment_type               0
product                    0
education_type             0
created_time               0
course_duration        15434
months_of_study        18984
initial_amount_paid    15762
offer_total_amount     15742
contact_id                47
city                       0
level_of_deutsch           0
is_student                 0
dtype: int64

### Проверяем, все ли контакты из Deals есть в таблице Contacts. 
То есть мы ищем такие contact_id, которые есть в Deals, но которых нет в Contacts.
если "0", то все сделки с contact_id корректно связаны с Contacts.

In [550]:
contacts_clean_path = r"C:\Users\Admin\Desktop\ICH\Учеба, курсы в записи, домашки\Финальный проект\дополнительные файлы\contacts_clean.xlsx"

contacts_clean = pd.read_excel(contacts_clean_path, dtype={"id": "string"})
contacts_clean["id"] = contacts_clean["id"].astype("string").str.strip()
unmatched_contact_ids = (set(deals_clean["contact_id"].dropna()) - set(contacts_clean["id"].dropna()))

len(unmatched_contact_ids)

0

# Итоговая проверка после очистки Deals

In [551]:
deals_clean_info = pd.DataFrame({
    "column": deals_clean.columns,
    "non_null": deals_clean.notna().sum().values,
    "missing": deals_clean.isna().sum().values,
    "missing_pct": (deals_clean.isna().mean().values * 100).round(2),
    "dtype": deals_clean.dtypes.astype(str).values,
    "unique_values": deals_clean.nunique(dropna=True).values})

deals_clean_info

,column,non_null,missing,missing_pct,dtype,unique_values
0,id,19824,0,0.00,string,19824
1,deal_owner_name,19795,29,0.15,object,27
2,closing_date,13138,6686,33.73,datetime64[ns],356
3,quality,17588,2236,11.28,object,5
4,stage,19824,0,0.00,object,13
5,lost_reason,14355,5469,27.59,object,21
6,page,19824,0,0.00,object,33
7,campaign,19824,0,0.00,object,154
8,sla,14997,4827,24.35,float64,12946
9,content,19824,0,0.00,object,185


In [552]:
# Проверяем дубликаты
print("Полных дублей:", deals_clean.duplicated().sum())
print("Дублей по id:", deals_clean["id"].duplicated().sum())

Полных дублей: 0
Дублей по id: 0


In [553]:
# Проверяем ключевые показатели

deals_cleaning_summary = pd.DataFrame({
    "metric": [
        "Строк после очистки",
        "Столбцов после очистки",
        "Уникальных id",
        "Пропусков в id",
        "Уникальных contact_id",
        "Студентов / buyer",
        "Уникальных менеджеров",
        "Уникальных stage",
        "Уникальных source",
        "Уникальных product",
        "Минимальная дата создания",
        "Максимальная дата создания",
        "Пропусков всего"],
    "value": [
        len(deals_clean),
        deals_clean.shape[1],
        deals_clean["id"].nunique(),
        deals_clean["id"].isna().sum(),
        deals_clean["contact_id"].nunique(),
        deals_clean["is_student"].sum(),
        deals_clean["deal_owner_name"].nunique(),
        deals_clean["stage"].nunique(),
        deals_clean["source"].nunique(),
        deals_clean["product"].nunique(),
        deals_clean["created_time"].min(),
        deals_clean["created_time"].max(),
        deals_clean.isna().sum().sum()]})

deals_cleaning_summary

,metric,value
0,Строк после очистки,19824
1,Столбцов после очистки,24
2,Уникальных id,19824
3,Пропусков в id,0
4,Уникальных contact_id,16985
5,Студентов / buyer,840
6,Уникальных менеджеров,27
7,Уникальных stage,13
8,Уникальных source,13
9,Уникальных product,6


In [554]:
# Проверяем деньги по студентам
students_money_summary = pd.DataFrame({
    "metric": [
        "Студентов всего",
        "Студентов с initial_amount_paid",
        "Студентов без initial_amount_paid",
        "Сумма initial_amount_paid по студентам",
        "Средний initial_amount_paid по студентам",
        "Медианный initial_amount_paid по студентам"],
    "value": [
        deals_clean["is_student"].sum(),
        deals_clean.loc[deals_clean["is_student"], "initial_amount_paid"].notna().sum(),
        deals_clean.loc[deals_clean["is_student"], "initial_amount_paid"].isna().sum(),
        deals_clean.loc[deals_clean["is_student"], "initial_amount_paid"].sum(),
        deals_clean.loc[deals_clean["is_student"], "initial_amount_paid"].mean(),
        deals_clean.loc[deals_clean["is_student"], "initial_amount_paid"].median()]})

students_money_summary

,metric,value
0,Студентов всего,840.000000
1,Студентов с initial_amount_paid,840.000000
2,Студентов без initial_amount_paid,0.000000
3,Сумма initial_amount_paid по студентам,950350.000000
4,Средний initial_amount_paid по студентам,1131.369048
5,Медианный initial_amount_paid по студентам,1000.000000


In [555]:
# Проверяем, остались ли случаи initial_amount_paid > offer_total_amount у студентов
students_money_errors = deals_clean[
    deals_clean["is_student"]
    & deals_clean["initial_amount_paid"].notna()
    & deals_clean["offer_total_amount"].notna()
    & (deals_clean["initial_amount_paid"] > deals_clean["offer_total_amount"])]

students_money_errors.shape

(0, 24)

## Вывод после итоговой проверки:
* После очистки проверены типы данных, пропуски, дубликаты, ключевые ID и основные показатели по студентам.
* В финальный файл не включались технические вспомогательные столбцы, которые использовались только на этапе очистки. Основной новый признак — `is_student`, который показывает, что у клиента заполнен `months_of_study`, то есть он фактически начал обучение.

## Сохраняем очищенный Deals и проверочные таблицы

In [556]:
output_dir = r"C:\Users\Admin\Desktop\ICH\Учеба, курсы в записи, домашки\Финальный проект\дополнительные файлы"
os.makedirs(output_dir, exist_ok=True)
deals_clean.to_csv(os.path.join(output_dir, "deals_clean.csv"), index=False, encoding="utf-8-sig")
deals_clean.to_excel(os.path.join(output_dir, "deals_clean.xlsx"), index=False)
deals_cleaning_summary.to_csv(os.path.join(output_dir, "deals_cleaning_summary.csv"), index=False, encoding="utf-8-sig")
students_money_summary.to_csv(os.path.join(output_dir, "deals_students_money_summary.csv"), index=False, encoding="utf-8-sig")
deals_clean_info.to_csv(os.path.join(output_dir, "deals_clean_info.csv"), index=False, encoding="utf-8-sig")

# Вывод по очистке Deals

Таблица Deals является основной таблицей проекта, так как именно в ней хранятся данные о потенциальных сделках, стадиях продаж, продуктах, оплатах, источниках лидов и признаках обучения.

При загрузке CRM ID (`id` и `contact_id`) были сохранены в строковом формате, чтобы избежать потери точности при дальнейших объединениях с Contacts и Calls.

В ходе очистки были удалены технически пустые строки без `id`, проверены полные дубли, дубли по `id`, бизнес-дубли и строки, помеченные как дубликаты в `lost_reason`. Строки с `lost_reason = Duplicate` удалялись только при отсутствии признака обучения, чтобы не потерять фактических студентов.

Столбцы `created_time` и `closing_date` были приведены к формату datetime. Ошибки, где `closing_date` была раньше `created_time`, были обработаны отдельно. Для сделок со статусом `Lost` некорректная дата закрытия была заменена на дату создания сделки, так как такие сделки были закрыты как потерянные, но дата закрытия была записана ошибочно.

Столбец `sla` был преобразован в числовой формат минут. Пропуски в SLA не заполнялись медианой, чтобы не искажать фактическое время реакции менеджеров.

Денежные поля `initial_amount_paid` и `offer_total_amount` были очищены от лишних символов и приведены к числовому формату. Пропуски в денежных полях не заменялись на 0, так как пропуск означает неизвестную или незафиксированную сумму, а не нулевую оплату. Для студентов были проверены случаи, где первый платеж больше общей суммы предложения; очевидные перестановки значений были исправлены.

Ключевым признаком покупателя был выбран `months_of_study` (Исходя из ответов на часто задаваемые вопросы, которые являются приложением к проекту). Если `months_of_study` заполнен, клиент считается студентом/buyer, так как по условиям проекта это означает, что человек фактически начал обучение. На основе этого был создан столбец `is_student`.

Продуктовые и маркетинговые поля были частично дозаполнены по истории того же контакта (`contact_id`), если у этого клиента в другой сделке уже было известно соответствующее значение. Оставшиеся пропуски в категориальных полях были заполнены значением `Unknown`, чтобы строки не выпадали из группировок. `months_of_study` не заполнялся искусственно, так как он является основным признаком факта обучения.

Поле `city` было нормализовано: убраны лишние пробелы и дополнительная информация после запятой. Часть пропусков была дозаполнена по истории контакта, оставшиеся значения заполнены как `Unknown`.

Поле `level_of_deutsch` было приведено к единому формату уровней немецкого языка. Нераспознанные или отсутствующие значения были отнесены к `Unknown`.

Перед сохранением была проведена ревизия столбцов. В финальный файл не включались технические вспомогательные поля, которые использовались только на этапе очистки. В итоговый датасет добавлен только один новый ключевой признак — `is_student`.

Очищенный Deals подготовлен и будет использован для дальнейшего анализа воронки продаж, эффективности менеджеров, источников лидов, продуктов, выручки и юнит-экономики.